# w2v-bert-tpu: speech recognition demo

Transcribes Ukrainian speech on a TPU v5e with
[w2v-bert-tpu](https://github.com/PositiveLoss/w2v-bert-tpu), using the fine-tuned
[`speech-uk/w2v-bert-v3`](https://huggingface.co/speech-uk/w2v-bert-v3) model (w2v-BERT 2.0 with a
CTC head) and the example clips from the
[Yehor/w2v-bert-uk-v3](https://huggingface.co/spaces/Yehor/w2v-bert-uk-v3) Space.

**Runtime:** Colab TPU v5e with TorchTPU (`torch_tpu`) preinstalled. Run the cells from a fresh
runtime, top to bottom.

## 1. Install

In [ ]:
import importlib.metadata as md
import subprocess
import sys


def version(pkg):
    try:
        return md.version(pkg)
    except md.PackageNotFoundError:
        return None


# Clone the code.
subprocess.run(["git", "clone", "-q", "https://github.com/PositiveLoss/w2v-bert-tpu"], check=False)

# Python packages needed for inference. sentencepiece decodes the model's tokens into text.
packages = ["transformers", "soundfile", "huggingface_hub", "sentencepiece"]
# jax compiles the Pallas kernels. libtpu 0.0.50 needs jax 0.11.2: newer jax refuses it.
if version("libtpu") == "0.0.50":
    packages += ["jax==0.11.2", "jaxlib==0.11.2"]
elif version("jax") is None:
    packages += ["jax"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

# Use the package straight from the clone (no restart needed).
sys.path.insert(0, "w2v-bert-tpu/src")

for pkg in ["torch", "torch_tpu", "libtpu", "jax", "jaxlib", "transformers"]:
    print(f"{pkg:13} {version(pkg)}")

## 2. Load the model and tokenizer

`JAX_PLATFORMS=cpu` must be set before anything imports JAX: JAX only compiles the kernels,
and TorchTPU owns the TPU. Only one process can use the TPU, so close other notebooks that use it.

`load()` reads the checkpoint's config and builds `Wav2Vec2BertForCTC`, the speech recognizer.

In [ ]:
import os

os.environ["JAX_PLATFORMS"] = "cpu"

import time

import sentencepiece as spm
import torch
from huggingface_hub import hf_hub_download

import w2vbert_tpu

MODEL = "speech-uk/w2v-bert-v3"

# Downloads the weights (2.4 GB), converts to BF16, moves to the TPU and sets up torch.compile.
model = w2vbert_tpu.load(MODEL)
print(type(model).__name__)

tokenizer = spm.SentencePieceProcessor(model_file=hf_hub_download(MODEL, "tokenizer.model"))
BLANK_ID, UNK_ID = 0, 1  # CTC blank (<pad>) and <unk>

## 3. Download the example audio

In [ ]:
from IPython.display import Audio, display

files = [
    hf_hub_download("Yehor/w2v-bert-uk-v3", f"example_{i}.wav", repo_type="space")
    for i in range(1, 7)
]


def load_clip(path):
    audio, sr = w2vbert_tpu.load_audio(path)  # mono, 16 kHz
    audio = audio - audio.mean()  # same peak normalization as the Space
    return audio / audio.abs().max().clamp_min(1e-8)


clips = [load_clip(f) for f in files]
for path, audio in zip(files, clips):
    print(f"{os.path.basename(path)}: {audio.numel() / 16000:.2f} s")
display(Audio(clips[0].numpy(), rate=16000))

## 4. Transcribe

`transcribe()` takes a list of waveforms, pads them to a fixed size (`pad_to_bucket`), runs the
model, and decodes the most likely token per frame (greedy CTC decoding).

Every new input shape is compiled once (8–30 s). Padding keeps the number of shapes small: all
six clips fit the same 512-frame bucket, so after the first call the rest are fast.

In [ ]:
def transcribe(waveforms):
    inputs = w2vbert_tpu.extract_features(waveforms, sampling_rate=16000)
    batch = w2vbert_tpu.pad_to_bucket(inputs["input_features"], inputs["attention_mask"])
    with torch.inference_mode():
        logits = model(
            input_features=batch.input_features.to("tpu", torch.bfloat16),
            attention_mask=batch.attention_mask.to("tpu"),
        ).logits
    # The model's adapter halves the frame rate; count the real (non-padding) logit frames.
    lengths = w2vbert_tpu.ctc_output_lengths(model.config, inputs["attention_mask"].sum(-1))
    token_ids = w2vbert_tpu.ctc_greedy_decode(
        logits, lengths, blank_id=BLANK_ID, ignore_ids=[UNK_ID]
    )
    return [tokenizer.decode_ids(ids).strip() for ids in token_ids]


for path, audio in zip(files, clips):
    t0 = time.perf_counter()
    (text,) = transcribe([audio])
    ms = (time.perf_counter() - t0) * 1000
    print(f"{os.path.basename(path)} ({ms:,.0f} ms): {text}")

## 5. Transcribe all clips as one batch

Clips of different lengths go in one batch; the attention mask keeps padding out of the result.
The first call compiles the batch-6 shape, the second shows the real speed.

In [ ]:
for run in ("first (compiles)", "second"):
    t0 = time.perf_counter()
    texts = transcribe(clips)
    print(f"{run}: {(time.perf_counter() - t0) * 1000:,.0f} ms for {len(clips)} clips")

for path, text in zip(files, texts):
    print(f"{os.path.basename(path)}: {text}")